# Lecture 06 — Modern CNN Architectures and Transfer Learning

> *ResNet, and why fine-tuning beats training from scratch almost every time.*

**What this notebook does**

Implement a residual block and a small ResNet; reproduce the degradation problem by training 8- and 22-layer plain networks and watching the deeper one fail to fit its own training data; then run the transfer-learning comparison: a source model pre-trained on one shape subset, transferred to a 100-example target task, against training from scratch.

**Data:** `shapes_32.npz, shapes_imagefolder/`

---

### How to use this notebook

**This notebook is your workbook and your submission.** Everything you need is
here — you do not need to open any other file.

1. Run the cells in order, top to bottom. The sections match the lecture slides.
2. Worked cells are there to be **read**, not skimmed. They build the ideas the
   tasks assume.
3. Cells marked **📝 TODO** are yours. Write your code in the empty cell
   underneath, and your written answer in a markdown cell after that.
4. When you are done, restart the kernel and run everything once more to check it
   works from clean.

Nothing here needs a GPU. If a cell is slow on your machine, reduce the epoch
count at the top of that section and say so in your write-up.

*(A printable copy of the tasks and the marking rubric is in
`../tasks/Lecture_06_Tasks.md`, but the work itself belongs here.)*

In [ ]:
# --- setup: make the repository root importable -------------------------
import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / "dlcourse").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from dlcourse import (load_shapes, load_digits_npz, load_detection,
                      load_segmentation, load_ssl_pool, load_sentiment,
                      load_corpus, load_captions,
                      train, evaluate, set_seed, count_parameters,
                      show_grid, plot_history, plot_confusion,
                      show_boxes, show_masks)

set_seed(0)
# Leave a core or two for the rest of the machine. More threads than cores
# makes training slower, not faster.
torch.set_num_threads(max(1, min(4, (os.cpu_count() or 4) - 1)))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("repo root :", ROOT)
print("torch     :", torch.__version__)
print("device    :", DEVICE)

## 0. Setup

In [ ]:
data = load_shapes(normalize=True)
X_train, y_train = data["train"]
X_val,   y_val   = data["val"]
X_test,  y_test  = data["test"]
CLASSES = data["classes"]
print("train", tuple(X_train.shape), "classes", CLASSES)

## 1. The degradation problem

In 2015 He et al. reported something that looked like a bug: a 56-layer plain
network had **higher training error** than a 20-layer one.

That is not overfitting — overfitting means higher *test* error with *lower*
training error. This is an optimisation failure. The deeper network could match
the shallow one exactly by learning the identity in its extra layers, and gradient
descent does not find that solution.

We reproduce it below at laptop scale. Two details make it visible:

- **Depth**: 22 layers versus 8.
- **No BatchNorm.** This is the setting the problem was discovered in, and it is
  the honest one: BatchNorm partially masks degradation, so leaving it in would
  let us claim a result the experiment does not show. You will measure BatchNorm's
  effect yourself in the stretch task.

In [ ]:
class Block(nn.Module):
    """One block, with the skip connection as a switch.

    Everything else is identical between the two variants, so any difference in
    the results is attributable to the skip and nothing else.
    """

    def __init__(self, channels, residual, use_bn=False):
        super().__init__()
        self.residual = residual
        self.c1 = nn.Conv2d(channels, channels, 3, padding=1, bias=not use_bn)
        self.c2 = nn.Conv2d(channels, channels, 3, padding=1, bias=not use_bn)
        self.b1 = nn.BatchNorm2d(channels) if use_bn else nn.Identity()
        self.b2 = nn.BatchNorm2d(channels) if use_bn else nn.Identity()

    def forward(self, x):
        out = F.relu(self.b1(self.c1(x)))
        out = self.b2(self.c2(out))
        return F.relu(out + x) if self.residual else F.relu(out)


def deep_net(n_blocks, residual, use_bn=False, width=24, seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(
        # Stride-2 stem: the stacked blocks run at 16x16 rather than 32x32, which
        # is 4x cheaper. Degradation is about depth, not resolution.
        nn.Conv2d(3, width, 3, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(width), nn.ReLU(),
        *[Block(width, residual, use_bn) for _ in range(n_blocks)],
        nn.AdaptiveMaxPool2d(2), nn.Flatten(),
        nn.Linear(width * 4, len(CLASSES)),
    )

print("8-layer plain   :", count_parameters(deep_net(3, False)), "parameters")
print("22-layer plain  :", count_parameters(deep_net(10, False)), "parameters")

In [ ]:
# ~4 minutes. Four networks. A small training subset keeps the *optimisation*
# difficulty -- not the amount of data -- as the thing being measured.
SUB = 1000
Xs, ys = X_train[:SUB], y_train[:SUB]
EPOCHS = 8
DEPTHS = [3, 10]                      # blocks -> 8 and 22 layers

degradation = {}
for residual in (False, True):
    for depth in DEPTHS:
        set_seed(0)
        model = deep_net(depth, residual)
        h = train(model, (Xs, ys), (X_val, y_val), epochs=EPOCHS,
                  lr=1e-3, batch_size=128, verbose=False)
        _, train_acc = evaluate(model, (Xs, ys))
        key = ("residual" if residual else "plain", depth * 2 + 2)
        degradation[key] = (h.train_loss[-1], train_acc, h.val_acc[-1])
        print(f"{key[0]:<9} {key[1]:>2} layers   "
              f"train_loss {h.train_loss[-1]:.4f}   "
              f"train_acc {train_acc:.3f}   val_acc {h.val_acc[-1]:.3f}")

In [ ]:
layer_counts = [d * 2 + 2 for d in DEPTHS]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.2))
for kind, colour in [("plain", "#c0392b"), ("residual", "#1b866b")]:
    a1.plot(layer_counts, [degradation[(kind, d)][0] for d in layer_counts],
            marker="o", lw=2, label=kind, color=colour)
    a2.plot(layer_counts, [degradation[(kind, d)][1] for d in layer_counts],
            marker="o", lw=2, label=kind, color=colour)
a1.set_xlabel("depth (layers)"); a1.set_ylabel("final TRAINING loss")
a1.set_title("Training loss vs depth")
a2.set_xlabel("depth (layers)"); a2.set_ylabel("TRAINING accuracy")
a2.set_title("Training accuracy vs depth")
a2.axhline(1 / len(CLASSES), color="k", ls=":", lw=1, label="chance")
for a in (a1, a2):
    a.set_xticks(layer_counts); a.grid(alpha=.3); a.legend()
fig.suptitle("Both axes are TRAINING metrics — this is optimisation failing, "
             "not overfitting")
plt.tight_layout(); plt.show()

plain_delta = degradation[("plain", 22)][1] - degradation[("plain", 8)][1]
res_delta = degradation[("residual", 22)][1] - degradation[("residual", 8)][1]
print(f"going from 8 to 22 layers changes TRAINING accuracy by:")
print(f"  plain    {plain_delta:+.3f}   <- deeper is worse")
print(f"  residual {res_delta:+.3f}   <- deeper is better")

### Read that carefully

Both plots show **training** metrics. The 22-layer plain network cannot even fit
the data it was trained on — it sits near chance. That is not a generalisation
problem that more data would fix. Gradient descent simply fails to find a good
solution.

And the deeper plain network is strictly *more expressive* than the shallow one:
it could copy the 8-layer network and set its extra blocks to the identity. The
solution exists. The optimiser does not reach it.

### Why the shortcut fixes it

A plain block must learn its whole mapping `H(x)` from scratch. If the best thing
it could do is nothing at all, it has to learn the identity — and a stack of
convolutions and ReLUs is surprisingly bad at representing the identity exactly.

A residual block computes `H(x) = F(x) + x`. To do nothing it only needs
`F(x) = 0`, which is easy: push the weights toward zero. **The identity is the
default behaviour, and the block learns the deviation from it.**

The second benefit is the one you measured in Lecture 3:
`d(x + F(x))/dx = 1 + dF/dx`. That `1` gives the gradient a path to the early
layers that skips every multiplication on the way.

### What BatchNorm does to this picture

Add BatchNorm to both block types and the collapse largely disappears — the
22-layer plain network trains, just slightly worse than the 8-layer one. BatchNorm
substantially mitigates degradation; residual connections remove it, and keep
working at depths where BatchNorm alone stops being enough.

That is stretch Task 6: run it with `use_bn=True` and compare.

## 2. A proper ResNet

Real ResNets change channel count and spatial size as they go, so the shortcut
needs a 1x1 projection whenever the shapes do not line up. That detail is where
most from-scratch implementations go wrong.

In [ ]:
class BasicBlock(nn.Module):
    """ResNet basic block with an optional projection shortcut."""
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        # Identity only works when shape and channels match; otherwise project.
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


# Check both paths.
same = BasicBlock(16, 16, stride=1)
down = BasicBlock(16, 32, stride=2)
t = torch.randn(2, 16, 32, 32)
print("identity shortcut :", tuple(t.shape), "->", tuple(same(t).shape))
print("projection shortcut:", tuple(t.shape), "->", tuple(down(t).shape))
print("\nshortcut type when shapes match    :", type(same.shortcut).__name__)
print("shortcut type when they do not     :", type(down.shortcut).__name__)

In [ ]:
class SmallResNet(nn.Module):
    """ResNet-style network sized for 32x32 inputs."""

    def __init__(self, blocks_per_stage=2, widths=(16, 32, 64), n_classes=4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, widths[0], 3, padding=1, bias=False),
            nn.BatchNorm2d(widths[0]), nn.ReLU(),
        )
        stages, in_ch = [], widths[0]
        for i, w in enumerate(widths):
            for b in range(blocks_per_stage):
                stride = 2 if (b == 0 and i > 0) else 1
                stages.append(BasicBlock(in_ch, w, stride))
                in_ch = w
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_ch, n_classes))

    def forward(self, x):
        return self.head(self.stages(self.stem(x)))

set_seed(0)
resnet = SmallResNet()
print(f"parameters: {count_parameters(resnet):,}")
print("output shape:", tuple(resnet(torch.randn(2, 3, 32, 32)).shape))

In [ ]:
# ~2 minutes.
set_seed(0)
resnet = SmallResNet()
hist = train(resnet, (X_train, y_train), (X_val, y_val),
             epochs=6, lr=2e-3, batch_size=128)
_, resnet_acc = evaluate(resnet, (X_test, y_test))
print(f"\nSmallResNet test accuracy: {resnet_acc:.4f}  "
      f"({count_parameters(resnet):,} parameters)")

In [ ]:
plot_history(hist, "SmallResNet on shapes")
plt.show()

## 3. Transfer learning

The real test of transfer learning is a target task with very few labels — that is
the situation where it matters.

The setup: pre-train on **circle vs square**, then transfer to **triangle vs star**
with only 100 labelled examples. Three strategies, same budget.

In [ ]:
# Split the dataset into a source task and a target task with disjoint classes.
src_mask_tr = (y_train == 0) | (y_train == 1)      # circle, square
tgt_mask_tr = (y_train == 2) | (y_train == 3)      # triangle, star
tgt_mask_te = (y_test == 2) | (y_test == 3)

Xsrc, ysrc = X_train[src_mask_tr], y_train[src_mask_tr]          # labels 0/1
Xtgt_all, ytgt_all = X_train[tgt_mask_tr], y_train[tgt_mask_tr] - 2
Xtgt_te,  ytgt_te  = X_test[tgt_mask_te],  y_test[tgt_mask_te] - 2

N_LABELS = 100
set_seed(0)
pick = torch.randperm(len(Xtgt_all))[:N_LABELS]
Xtgt, ytgt = Xtgt_all[pick], ytgt_all[pick]

print(f"source task (circle vs square) : {len(Xsrc)} examples")
print(f"target task (triangle vs star) : {len(Xtgt)} labelled, "
      f"{len(Xtgt_te)} test")

In [ ]:
# Pre-train the source model. ~60 s.
set_seed(0)
source_model = SmallResNet(n_classes=2)
train(source_model, (Xsrc, ysrc), epochs=5, lr=2e-3, batch_size=128, verbose=False)
_, src_acc = evaluate(source_model, (Xsrc, ysrc))
print(f"source task accuracy: {src_acc:.3f}")

In [ ]:
import copy

def target_model_from(source, mode):
    """mode: 'scratch' | 'frozen' | 'finetune'."""
    if mode == "scratch":
        set_seed(1)
        return SmallResNet(n_classes=2), 2e-3

    model = copy.deepcopy(source)
    model.head[-1] = nn.Linear(model.head[-1].in_features, 2)   # fresh head
    if mode == "frozen":
        for p in model.stem.parameters():
            p.requires_grad = False
        for p in model.stages.parameters():
            p.requires_grad = False
        return model, 2e-3
    return model, 2e-4          # fine-tune: 10x lower learning rate

transfer_results = {}
for mode in ("scratch", "frozen", "finetune"):
    set_seed(2)
    model, lr = target_model_from(source_model, mode)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=lr)
    train(model, (Xtgt, ytgt), epochs=25, optimizer=opt,
          batch_size=32, verbose=False)
    _, acc = evaluate(model, (Xtgt_te, ytgt_te))
    transfer_results[mode] = acc
    trainable = sum(p.numel() for p in params)
    print(f"{mode:<10} target test acc {acc:.3f}   ({trainable:,} trainable parameters)")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.2))
labels = ["from scratch", "frozen backbone\n+ new head", "full fine-tune\n(lr/10)"]
vals = [transfer_results[m] for m in ("scratch", "frozen", "finetune")]
bars = ax.barh(labels, vals, color=["#c0392b", "#e67e22", "#1b866b"])
for i, v in enumerate(vals):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=11)
ax.set_xlim(0, 1.05); ax.set_xlabel("target-task test accuracy")
ax.set_title(f"Transfer learning with only {N_LABELS} labelled target examples")
plt.tight_layout(); plt.show()

### Choosing a strategy

| Target data | Domain distance | What to do |
|---|---|---|
| Small | Close | Freeze the backbone, train a linear head |
| Large | Close | Fine-tune everything |
| Small | Far | Fine-tune the middle layers; very late features may not transfer |
| Large | Far | Fine-tuning still usually converges faster than random init |

The learning rate for fine-tuning is conventionally 10x lower than you would use
from scratch. The pre-trained weights are already good; a large learning rate
destroys them in the first few batches — a failure mode with its own name,
*catastrophic forgetting*.

## 4. How much should you unfreeze?

Not a binary choice. Sweep it.

In [ ]:
def unfreeze_last_k(source, k):
    model = copy.deepcopy(source)
    model.head[-1] = nn.Linear(model.head[-1].in_features, 2)
    blocks = list(model.stages)
    for p in model.stem.parameters():
        p.requires_grad = False
    for i, block in enumerate(blocks):
        trainable = i >= len(blocks) - k
        for p in block.parameters():
            p.requires_grad = trainable
    return model

n_blocks = len(list(source_model.stages))
sweep = {}
for k in range(0, n_blocks + 1, 2):
    set_seed(3)
    model = unfreeze_last_k(source_model, k)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=5e-4)
    train(model, (Xtgt, ytgt), epochs=25, optimizer=opt, batch_size=32, verbose=False)
    _, acc = evaluate(model, (Xtgt_te, ytgt_te))
    sweep[k] = (acc, sum(p.numel() for p in params))
    print(f"unfrozen blocks: {k}/{n_blocks}   acc {acc:.3f}   "
          f"trainable {sweep[k][1]:,}")

In [ ]:
ks = list(sweep)
accs = [sweep[k][0] for k in ks]
trainables = [sweep[k][1] for k in ks]

fig, ax1 = plt.subplots(figsize=(8, 4.2))
ax1.plot(ks, accs, marker="o", color="#1b866b", lw=2)
ax1.set_xlabel("number of unfrozen backbone blocks")
ax1.set_ylabel("target test accuracy", color="#1b866b")
ax1.grid(alpha=.3)
ax2 = ax1.twinx()
ax2.plot(ks, trainables, marker="s", color="#7f8c8d", ls="--")
ax2.set_ylabel("trainable parameters", color="#7f8c8d")
ax1.set_title("How much to unfreeze — accuracy against cost")
plt.tight_layout(); plt.show()

## 5. Architecture building blocks worth knowing

Two ideas you will meet constantly: the 1x1 bottleneck, and depthwise separable
convolution. Both are about buying the same receptive field for fewer operations.

In [ ]:
def count(m):
    return sum(p.numel() for p in m.parameters())

C = 256
plain = nn.Sequential(nn.Conv2d(C, C, 3, padding=1), nn.Conv2d(C, C, 3, padding=1))
bottleneck = nn.Sequential(
    nn.Conv2d(C, C // 4, 1),                    # reduce
    nn.Conv2d(C // 4, C // 4, 3, padding=1),    # process cheaply
    nn.Conv2d(C // 4, C, 1),                    # expand
)
depthwise = nn.Sequential(
    nn.Conv2d(C, C, 3, padding=1, groups=C),    # one filter per channel
    nn.Conv2d(C, C, 1),                         # mix channels
)

print(f"{'block':<32}{'parameters':>14}{'ratio':>10}")
print("-" * 56)
base = count(plain)
for name, m in [("plain 3x3 + 3x3", plain),
                ("1x1 -> 3x3 -> 1x1 bottleneck", bottleneck),
                ("depthwise separable", depthwise)]:
    print(f"{name:<32}{count(m):>14,}{count(m)/base:>10.2f}x")

x = torch.randn(1, C, 16, 16)
for name, m in [("plain", plain), ("bottleneck", bottleneck), ("depthwise", depthwise)]:
    print(f"\n{name} output: {tuple(m(x).shape)}")

---

## 📝 TODO — Task 1: Residual block from scratch

Implement `BasicBlock` with two 3x3 convolutions, BatchNorm and a skip connection,
including the 1x1 projection needed when channel count or stride changes. Verify
output shapes for both cases.

**Deliverable:** working code in the cell below, plus the number, table or plot the
task asks for. Where you are asked to explain something, add a markdown cell
underneath and answer in two or three sentences.

In [ ]:
# ---- YOUR SOLUTION ----

---

## 📝 TODO — Task 2: Reproduce the degradation problem

Train an 8-layer and a 22-layer plain CNN **without BatchNorm**. Show the deeper
one has higher **training** loss. Add skip connections to both and show the
ordering reverses.

Be explicit in your write-up that both numbers are *training* metrics — that is
what separates degradation from overfitting.

**Deliverable:** working code in the cell below, plus the number, table or plot the
task asks for. Where you are asked to explain something, add a markdown cell
underneath and answer in two or three sentences.

In [ ]:
# ---- YOUR SOLUTION ----

---

## 📝 TODO — Task 3: Small ResNet on shapes

Assemble a ResNet-style network from your blocks. Reach at least 99% test accuracy
and report parameter count and training time against the Lecture 5 CNN.

**Deliverable:** working code in the cell below, plus the number, table or plot the
task asks for. Where you are asked to explain something, add a markdown cell
underneath and answer in two or three sentences.

In [ ]:
# ---- YOUR SOLUTION ----

---

## 📝 TODO — Task 4: Feature extraction versus fine-tuning

Pre-train on circle/square only. Transfer to a triangle/star task with just 100
labelled examples, three ways: from scratch, frozen backbone, full fine-tuning at
`lr/10`. Report all three accuracies.

**Deliverable:** working code in the cell below, plus the number, table or plot the
task asks for. Where you are asked to explain something, add a markdown cell
underneath and answer in two or three sentences.

In [ ]:
# ---- YOUR SOLUTION ----

---

## 📝 TODO — Task 5: How many layers to unfreeze

Sweep the number of unfrozen backbone blocks from 0 to all. Plot target-task
accuracy against that number and state where the curve flattens.

**Deliverable:** working code in the cell below, plus the number, table or plot the
task asks for. Where you are asked to explain something, add a markdown cell
underneath and answer in two or three sentences.

In [ ]:
# ---- YOUR SOLUTION ----

---

## 📝 TODO — Task 6: 1x1 bottleneck cost analysis *(stretch)*

Compare parameters and FLOPs for a plain 3x3-3x3 block against a 1x1-3x3-1x1
bottleneck of equal input/output width. Report the ratio and verify with a
forward-pass timing.

**Deliverable:** working code in the cell below, plus the number, table or plot the
task asks for. Where you are asked to explain something, add a markdown cell
underneath and answer in two or three sentences.

In [ ]:
# ---- YOUR SOLUTION ----

---

## 📝 TODO — Task 7: Depthwise separable convolution *(stretch)*

Implement it and substitute it into your ResNet. Report the accuracy lost and the
parameters saved.

**Deliverable:** working code in the cell below, plus the number, table or plot the
task asks for. Where you are asked to explain something, add a markdown cell
underneath and answer in two or three sentences.

In [ ]:
# ---- YOUR SOLUTION ----

---

## Checklist before you submit

All 5 core tasks are required. Each is marked **📝 TODO** above, in the
section it belongs to.

- [ ] **Task 1** — Residual block from scratch
- [ ] **Task 2** — Reproduce the degradation problem
- [ ] **Task 3** — Small ResNet on shapes
- [ ] **Task 4** — Feature extraction versus fine-tuning
- [ ] **Task 5** — How many layers to unfreeze

Optional, not marked:

- [ ] *Stretch 1* — What BatchNorm does to degradation
- [ ] *Stretch 2* — 1x1 bottleneck cost analysis
- [ ] *Stretch 3* — Depthwise separable convolution

### Final checks

- [ ] Restart the kernel and **Run All**. It completes with no errors.
- [ ] `set_seed(0)` runs before anything random.
- [ ] Every plot has axis labels and a title.
- [ ] Written answers are in markdown cells, not in code comments.
- [ ] This lecture's headline numbers are in your running `results.md` table.
- [ ] AI-assistant use is disclosed in the cell below.

Submit this notebook as `LASTNAME_FIRSTNAME_L06.ipynb`.

### AI assistance disclosure

*Replace this text: state which tools you used and for what. "None" is a valid answer.*